## Build and validate agent-governance docs for your repo — hands-on
Goal: turn your team's unwritten norms into two concrete files that AI coding assistants follow automatically, and add a small script that checks they stay healthy in PRs.

Roadmap — what we will do, step by step (plain-first, jargon-in-brackets):
- Pick a working repository (or create a throwaway demo one) and set up the notebook to write files into it.
- Create a repo-level instruction file for assistants (".github/copilot-instructions.md") that states purpose, scope, coding style, examples, and safety rules.
- Create an agent catalog ("AGENTS.md") that defines named personas, what they may do, their task templates, sample interactions, and hard boundaries.
- Write a small Python checker that verifies structure and sanity of both files, run it locally, and read the report.
- Commit your changes locally; pushing to remote is optional and off by default.

What you should understand afterward:
- How to encode Python repository conventions into precise rules assistants can follow (imports, typing, tests, and lint/language rules).
- How to structure agent personas/workflows so assistants act within boundaries.
- How to automate basic validation in CI (continuous integration) to catch omissions early.

Assumptions and likely gaps:
- Assumes you can read/write basic Python and Markdown, and you know git basics (clone/commit/push).
- Likely gap 1: how Python repos are usually laid out (src/, package root, tests/). We'll do a short primer.
- Likely gap 2: what a repo-level assistant instruction file actually governs. We'll make that concrete with examples.
- Likely gap 3: how to add simple CI validation hooks. We'll explain the idea and give a ready-to-use checker.

Note: This is an artifact-building lesson. Every step writes real files and validates them so you can SEE they work.

### Pipeline map — what we will build
```text
[Choose/Bootstrap Repo]
          |
          v
[Write .github/copilot-instructions.md]
          |
          v
[Quick-validate instructions (structure + examples)]
          |
          v
[Write AGENTS.md (personas, actions, templates, boundaries)]
          |
          v
[Quick-validate agents (sections + templates + interactions)]
          |
          v
[tools/validate_agent_docs.py]
          |
          v
[Run validator script -> JSON + summary]
          |
          v
[git add/commit (local); optional push]
```
Artifacts on disk: .github/copilot-instructions.md, AGENTS.md, tools/validate_agent_docs.py. The validator is the anchor you can re-use in CI.

In [ ]:
# Setup check — run me first
import sys, subprocess, shutil
from pathlib import Path

# Verify Python and external tools we rely on
MIN_PY = (3, 8)
if sys.version_info < MIN_PY:
    raise SystemExit(f"Python {MIN_PY[0]}.{MIN_PY[1]}+ required; found {sys.version.split()[0]}. Please use Python {MIN_PY[0]}.{MIN_PY[1]} or newer.")

if shutil.which("git") is None:
    raise SystemExit("Missing prerequisite: git (install from https://git-scm.com/downloads and ensure it's on PATH)")

# Print versions so you can confirm your environment
try:
    git_ver = subprocess.run(["git", "--version"], capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    git_ver = "git --version (unavailable)"
print("Setup OK — Python", sys.version.split()[0], "|", git_ver)

### Workspace selection
We will write files into a git repository. If you're already inside a repo, we will use it. If not, we will create a local demo repo named "agent-docs-demo" in the current directory so you can run the lesson end-to-end.

Stand-in note: the demo repo is created here in the notebook to give you a safe place to practice. It is not your real project; swap it for your own repo path when ready.

In [ ]:
# Choose or bootstrap a git repository to work in
import os, subprocess
from pathlib import Path

 def _is_git_repo(path: Path) -> bool:
    try:
        r = subprocess.run(["git", "rev-parse", "--is-inside-work-tree"], cwd=path, capture_output=True, text=True)
        return r.returncode == 0 and r.stdout.strip() == "true"
    except Exception:
        return False

# Prefer the current working directory if it's a git repo
cwd = Path.cwd()
if _is_git_repo(cwd):
    repo = cwd
else:
    repo = cwd / "agent-docs-demo"
    repo.mkdir(parents=True, exist_ok=True)
    # Initialize a minimal repo
    subprocess.run(["git", "init"], cwd=repo, check=True)
    # Configure a local username/email to allow commits without global config
    subprocess.run(["git", "config", "user.name", "Lesson Bot"], cwd=repo, check=False)
    subprocess.run(["git", "config", "user.email", "lesson@example.com"], cwd=repo, check=False)
    # Seed with a README so the first commit exists
    (repo / "README.md").write_text("# Demo repo for agent-governance lesson\n")
    subprocess.run(["git", "add", "README.md"], cwd=repo, check=True)
    subprocess.run(["git", "commit", "-m", "chore: initial commit [lesson]"], cwd=repo, check=True)

# Ensure standard folders exist
(repo / ".github").mkdir(parents=True, exist_ok=True)
(repo / "tools").mkdir(parents=True, exist_ok=True)

# Move into the repo for all subsequent operations
os.chdir(repo)
print(f"Using repository at: {repo}")
# Show current branch for context
subprocess.run(["git", "status", "-sb"], cwd=repo)

## Why an assistant-instructions file exists
- What it is: a repo-level contract for AI coding assistants. It turns vague prompts into precise, repeatable guidance tied to your project.
- Why it matters: assistants can only guess your layout, style, and safety rules unless you tell them. Central instructions remove guesswork and reduce risky suggestions.

Example of clarity gained:
- Ambiguous prompt: "Write a new API." Assistant might scaffold code under the wrong folder, skip tests, or ignore typing.
- With repo-level rule: "All Python code lives under src/<package>/; new endpoints require tests under tests/api/ and must pass pytest -q and mypy." Now assistants place files correctly and include tests by default.

## Python repo conventions to codify (primer)
- Layout: put importable code under src/<package_name>/ and tests under tests/. This avoids import path confusion and matches modern tooling.
- Tests: prefer pytest. Make the default command explicit (e.g., "pytest -q").
- Typing: adopt type hints and a strictness level (e.g., mypy). Assistants should include annotations on new/changed code.
- Lint/format: pick tools (e.g., ruff/flake8, black) and state expectations (e.g., "run ruff and black locally before PRs").
- Imports: use absolute imports from your package (src layout makes this unambiguous).

In [ ]:
# Artifact 1 — write .github/copilot-instructions.md
from textwrap import dedent
from pathlib import Path

copilot_text = dedent(
    """
    # Copilot Instructions for This Repository

    ## Purpose
    Provide precise, repository-specific guidance so AI coding assistants generate code, tests, and docs that follow our team's conventions and safety rules.

    ## Scope
    These instructions apply to Python code, tests, docs, and simple shell snippets in this repository. They define layout, style, testing, typing, and security constraints. Prefer explicit guidance over inference.

    ## Coding conventions
    ### Package layout
    - All importable Python code lives under `src/<package_name>/`.
    - All tests live under `tests/` and are runnable with `pytest -q`.

    ### Imports
    - Use absolute imports from our package (enabled by the `src/` layout). Avoid relative imports beyond a single `.` when refactoring.

    ### Typing
    - Add type hints to all public functions and new code. Maintain mypy compliance (run `mypy` locally or reason about types in changes).

    ### Tests
    - Every new behavior requires tests under `tests/`. Name files `test_*.py`. Provide focused unit tests. Assume CI runs `pytest` on PRs; design for determinism.

    ### Linting and formatting
    - Follow ruff/flake8 rules and Black formatting. Include fixes in the change. Avoid unused imports and variables.

    ## Examples
    Clear before/after examples assistants can mirror.

    #### Before
    ```python
    def add(a,b):return a+b
    ```

    #### After
    ```python
    from __future__ import annotations

    def add(a: int, b: int) -> int:
        """Add two integers."""
        return a + b
    ```

    Another example — correct placement under `src/` with tests under `tests/` using `pytest` and type checks via `mypy`.

    ## Security constraints
    - Never introduce hard-coded secrets or tokens; use environment variables and configuration files as appropriate.
    - Do not include credentials in code, comments, tests, or commit messages.
    - Do not propose commands that access private network resources or identity files. Prefer local, reproducible steps.

    ## Escalation
    If a task conflicts with these rules or requires privileged access:
    1) Stop and ask for clarification.
    2) Propose a safe plan with assumptions.
    3) Label any uncertainty explicitly in the PR description or comment.
    """
).strip() + "\n"

path_ci = Path(".github") / "copilot-instructions.md"
path_ci.write_text(copilot_text, encoding="utf-8")
print(f"Wrote {path_ci} ({len(copilot_text.splitlines())} lines)")

What was created
- File: .github/copilot-instructions.md
- What's inside: purpose, scope, detailed coding conventions (layout/imports/typing/tests/lint), examples with Before/After code fences, and security + escalation guidance.
- Why it matters: assistants now have repo-tied rules and concrete examples to mirror.

In [ ]:
# Validate copilot-instructions.md (structure + examples + tokens)
import re, json
from pathlib import Path

def validate_copilot_instructions(path: Path) -> dict:
    out = {"exists": path.exists(), "missing_headers": [], "before_after_ok": False,
           "has_code_fence": False, "tokens_present": [], "forbidden_hits": []}
    if not path.exists():
        return out
    text = path.read_text(encoding="utf-8")
    lines = text.splitlines()
    # Required headings (case-insensitive substring match in Markdown headings)
    required = ["Purpose", "Scope", "Coding conventions", "Examples", "Security constraints", "Escalation"]
    headers = [ln for ln in lines if ln.lstrip().startswith(("#", "##", "###", "####"))]
    for req in required:
        found = any(req.lower() in h.lower() for h in headers)
        if not found:
            out["missing_headers"].append(req)
    # Code fences
    fences = sum(1 for ln in lines if ln.strip().startswith("```") )
    out["has_code_fence"] = fences >= 1
    # Before/After headings with adjacent code blocks
    text_lower = text.lower()
    before_idx = text_lower.find("\n#### before")
    after_idx = text_lower.find("\n#### after")
    out["before_after_ok"] = before_idx != -1 and after_idx != -1 and after_idx > before_idx and fences >= 2
    # Repo tokens
    tokens = ["src/", "tests/", "pytest", "mypy"]
    out["tokens_present"] = [t for t in tokens if t.lower() in text_lower]
    # Forbidden patterns (only patterns that indicate dangerous content, not policy mentions)
    patterns = {
        "aws_access_key": re.compile(r"AKIA[0-9A-Z]{16}"),
        "private_key_block": re.compile(r"-----BEGIN (?:RSA|EC|OPENSSH|DSA)? ?PRIVATE KEY-----"),
        "password_assignment": re.compile(r"password\s*=", re.I),
        "ssh_identity_flag": re.compile(r"ssh\s+-i", re.I),
        "curl_internal_http": re.compile(r"curl\s+http://internal", re.I),
    }
    for name, rx in patterns.items():
        if rx.search(text):
            out["forbidden_hits"].append(name)
    out["passed"] = out["exists"] and not out["missing_headers"] and out["before_after_ok"] and out["has_code_fence"] and bool(out["tokens_present"]) and not out["forbidden_hits"]
    return out

res_ci = validate_copilot_instructions(Path(".github") / "copilot-instructions.md")
print(json.dumps(res_ci, indent=2))
assert res_ci.get("passed"), "copilot-instructions.md failed validation; see JSON above"

Interpretation
- Headers: all required sections should be found; if any are missing, add them as Markdown headings.
- Examples: look for Before/After headings with code fences — assistants mirror concrete patterns.
- Tokens: at least one of src/, tests/, pytest, mypy must be present so layout and tooling are explicit.
- Forbidden patterns: should be empty; the validator looks for actual risky patterns (e.g., password=), not policy mentions.

## Anatomy of AGENTS.md (personas and boundaries)
- What it is: a catalog of named assistant roles (agents) with responsibilities, allowed actions, templates, sample interactions, and hard boundaries.
- Why it matters: assistants act more safely and consistently when they have scoped roles and explicit do/don't lists.
- Structure per agent:
  - Persona header (e.g., "## Agent: CodeGen Engineer — Persona").
  - Responsibilities — what outcomes they own.
  - Allowed actions — the concrete operations they may take/suggest.
  - Task templates — parameterized prompts/checklists (e.g., {{issue_number}}) to reduce ambiguity.
  - Sample interactions — at least one request and one agent reply as code-fenced examples.
  - Boundaries and sensitive-data rules — explicit things to refuse or escalate.

In [ ]:
# Artifact 2 — write AGENTS.md
from textwrap import dedent
from pathlib import Path

agents_text = dedent(
    """
    # AGENTS — Named assistant personas and boundaries

    This document defines the agents used in this repository, their responsibilities, allowed actions, reusable task templates, sample interactions, and strict boundaries.

    ## Agent: CodeGen Engineer — Persona

    ### Responsibilities
    - Implement features and refactors under `src/<package_name>/`.
    - Maintain and extend tests under `tests/` so `pytest -q` passes.
    - Keep code typed and clean (mypy, ruff/flake8, Black).

    ### Allowed actions
    - Propose file additions/edits under `src/` and `tests/`.
    - Draft unit tests and fixtures; prefer small, focused tests.
    - Provide diffs/patches and a short rationale.
    - Suggest local commands for the developer to run (e.g., `pytest -q`, `mypy`), but do not execute them.

    ### Task templates
    - Implement feature `{{feature_name}}` touching module `{{module_path}}`:
      1) Update `src/{{package_name}}/{{module_path}}` with typed functions.
      2) Add tests in `tests/{{module_area}}/test_{{feature_name}}.py`.
      3) Ensure `pytest -q` and `mypy` succeed locally.
    - Refactor function `{{function_name}}` in `{{file_path}}`:
      - Keep public API; add/adjust tests.
      - Provide a minimal diff and reasoning.

    ### Sample interactions
    Request:
    ```text
    Please add a slugify utility to the strings module with tests.
    ```

    Agent reply:
    ```diff
    + src/acme/strings.py
    + def slugify(name: str) -> str:
    +     """Convert arbitrary text to a URL-safe slug."""
    +     # implementation here
    + tests/strings/test_slugify.py
    + def test_slugify_basic():
    +     assert slugify("Hello World!") == "hello-world"
    ```

    ### Boundaries and sensitive-data rules
    - Do not introduce or request credentials; use configuration and environment variables.
    - Do not propose commands that require access to private networks or identity files.
    - If unsure about layout or policy, ask for clarification before proceeding.

    ## Agent: Release Steward — Persona

    ### Responsibilities
    - Prepare changelogs and release notes.
    - Bump versions consistently and tag releases.

    ### Allowed actions
    - Propose edits to `CHANGELOG.md` and version files.
    - Draft release PR descriptions and checklists.

    ### Task templates
    - Draft release notes for version `{{version}}` using merged PRs from `{{since_tag}}` to `{{until_tag}}`.
    - Propose version bump from `{{old_version}}` to `{{new_version}}` with rationale.

    ### Sample interactions
    Request:
    ```text
    Prepare release notes for v1.4.0.
    ```

    Agent reply:
    ```markdown
    ## v1.4.0 — 2024-05-12
    - Feature: add slugify utility (PR #{{issue_number}})
    - Fix: handle Windows paths in loader
    ```

    ### Boundaries and sensitive-data rules
    - Do not fabricate changelog entries; cite PR numbers and commit hashes when available.
    - Escalate if versioning scheme is ambiguous.
    """
).strip() + "\n"

path_agents = Path("AGENTS.md")
path_agents.write_text(agents_text, encoding="utf-8")
print(f"Wrote {path_agents} ({len(agents_text.splitlines())} lines)")

What was created
- File: AGENTS.md at the repo root.
- What's inside: two agents (CodeGen Engineer, Release Steward) with required subheadings, parameterized task templates using {{placeholders}}, and paired sample interactions (request + reply) using code fences.
- Why it matters: assistants get scoped roles and explicit do/don't lists to act safely.

In [ ]:
# Validate AGENTS.md (agents + sections + templates + interactions)
import re, json
from pathlib import Path

def parse_agents_blocks(text: str):
    lines = text.splitlines()
    blocks = []
    current = None
    for ln in lines:
        if ln.startswith("## ") and "Agent:" in ln:
            if current:
                blocks.append(current)
            current = {"header": ln.strip(), "lines": []}
        elif current is not None:
            current["lines"].append(ln)
    if current:
        blocks.append(current)
    return blocks


def validate_agents_md(path: Path) -> dict:
    out = {"exists": path.exists(), "agent_count": 0, "agents": [], "passed": False}
    if not path.exists():
        return out
    text = path.read_text(encoding="utf-8")
    blocks = parse_agents_blocks(text)
    out["agent_count"] = len(blocks)
    overall = True
    for blk in blocks:
        body = "\n".join(blk["lines"]).lower()
        subs = {
            "Responsibilities": "responsibilities" in body,
            "Allowed actions": "allowed actions" in body,
            "Task templates": "task templates" in body,
            "Sample interactions": "sample interactions" in body,
            "Boundaries and sensitive-data rules": "boundaries" in body and "sensitive" in body,
        }
        missing = [k for k, ok in subs.items() if not ok]
        # Template placeholders
        has_template_placeholder = bool(re.search(r"\{\{[^}]+\}\}", body))
        # Request + Agent reply with code fences
        has_request = "request:" in body and body.count("```") >= 1
        has_reply = "agent reply:" in body and body.count("```") >= 2
        passed = (not missing) and has_template_placeholder and has_request and has_reply
        overall = overall and passed
        out["agents"].append({
            "header": blk["header"],
            "missing_sections": missing,
            "has_template_placeholder": has_template_placeholder,
            "has_request_and_reply": has_request and has_reply,
            "passed": passed,
        })
    out["passed"] = overall and out["agent_count"] >= 1
    return out

res_agents = validate_agents_md(Path("AGENTS.md"))
print(json.dumps(res_agents, indent=2))
assert res_agents.get("passed"), "AGENTS.md failed validation; see JSON above"

Interpretation
- Each agent must include all required subheadings; the JSON shows any missing ones.
- Templates: at least one {{placeholder}} should appear so tasks are parameterizable.
- Interactions: there should be a request and an agent reply, each shown as a code-fenced example.

## Validation design and CI hook (concept)
- Why automate: reviewers can miss structure gaps; a quick script catches them before PRs land.
- What to check:
  - Existence of both files.
  - Required headings (case-insensitive), and at least one code fence.
  - For instructions: presence of repo tokens (src/, tests/, pytest, mypy) and a Before/After example; scan for risky patterns (e.g., password=, private key blocks, AWS key shapes).
  - For agents: per-agent required sections, at least one {{placeholder}} in templates, and a request/reply pair with code blocks.
- How to use in CI: run the script in a job on every PR and fail fast if structure is broken. We'll produce a reusable tools/validate_agent_docs.py now.

In [ ]:
# Artifact 3 — write tools/validate_agent_docs.py (consolidated checker)
from pathlib import Path
from textwrap import dedent

validator_py = dedent(
    """
    #!/usr/bin/env python3
    import sys, re, json
    from pathlib import Path

    def load_text(p: Path) -> str:
        try:
            return p.read_text(encoding="utf-8")
        except FileNotFoundError:
            return ""

    def headings(md: str):
        return [ln.strip() for ln in md.splitlines() if ln.lstrip().startswith(("#","##","###","####"))]

    def has_code_fence(md: str, at_least=1):
        return sum(1 for ln in md.splitlines() if ln.strip().startswith("````") or ln.strip().startswith("```") ) >= at_least

    def validate_instructions(p: Path):
        text = load_text(p)
        req = ["Purpose", "Scope", "Coding conventions", "Examples", "Security constraints", "Escalation"]
        hdrs = headings(text)
        missing = [h for h in req if not any(h.lower() in H.lower() for H in hdrs)]
        tl = text.lower()
        before_after = ("\n#### before" in tl) and ("\n#### after" in tl) and has_code_fence(text, at_least=2)
        tokens = [t for t in ["src/", "tests/", "pytest", "mypy"] if t in tl]
        patterns = {
            "aws_access_key": re.compile(r"AKIA[0-9A-Z]{16}"),
            "private_key_block": re.compile(r"-----BEGIN (?:RSA|EC|OPENSSH|DSA)? ?PRIVATE KEY-----"),
            "password_assignment": re.compile(r"password\s*=", re.I),
            "ssh_identity_flag": re.compile(r"ssh\s+-i", re.I),
            "curl_internal_http": re.compile(r"curl\s+http://internal", re.I),
        }
        forbidden_hits = [name for name, rx in patterns.items() if rx.search(text)]
        return {
            "exists": p.exists(),
            "missing_headers": missing,
            "before_after_ok": before_after,
            "has_code_fence": has_code_fence(text),
            "tokens_present": tokens,
            "forbidden_hits": forbidden_hits,
            "passed": p.exists() and not missing and before_after and has_code_fence(text) and bool(tokens) and not forbidden_hits,
        }

    def parse_agents_blocks(text: str):
        lines = text.splitlines()
        blocks = []
        current = None
        for ln in lines:
            if ln.startswith("## ") and "Agent:" in ln:
                if current:
                    blocks.append(current)
                current = {"header": ln.strip(), "lines": []}
            elif current is not None:
                current["lines"].append(ln)
        if current:
            blocks.append(current)
        return blocks

    def validate_agents(p: Path):
        text = load_text(p)
        if not text:
            return {"exists": False, "agent_count": 0, "agents": [], "passed": False}
        blocks = parse_agents_blocks(text)
        agents = []
        overall = True
        for blk in blocks:
            body = "\n".join(blk["lines"]).lower()
            subs = {
                "Responsibilities": "responsibilities" in body,
                "Allowed actions": "allowed actions" in body,
                "Task templates": "task templates" in body,
                "Sample interactions": "sample interactions" in body,
                "Boundaries and sensitive-data rules": "boundaries" in body and "sensitive" in body,
            }
            missing = [k for k, ok in subs.items() if not ok]
            has_template_placeholder = bool(re.search(r"\{\{[^}]+\}\}", body))
            has_request = "request:" in body and body.count("```") >= 1
            has_reply = "agent reply:" in body and body.count("```") >= 2
            passed = (not missing) and has_template_placeholder and has_request and has_reply
            overall = overall and passed
            agents.append({
                "header": blk["header"],
                "missing_sections": missing,
                "has_template_placeholder": has_template_placeholder,
                "has_request_and_reply": has_request and has_reply,
                "passed": passed,
            })
        return {"exists": True, "agent_count": len(blocks), "agents": agents, "passed": overall and len(blocks) >= 1}

    def main():
        root = Path.cwd()
        p_instructions = root / ".github" / "copilot-instructions.md"
        p_agents = root / "AGENTS.md"
        res_i = validate_instructions(p_instructions)
        res_a = validate_agents(p_agents)
        report = {
            "copilot_instructions": res_i,
            "agents": res_a,
            "overall_passed": bool(res_i.get("passed")) and bool(res_a.get("passed")),
        }
        # Print JSON first for machine parsing
        print(json.dumps(report))
        # Then a human-readable summary
        print("\nValidation summary:")
        if not res_i["exists"]:
            print(f"- MISSING: {p_instructions}")
        else:
            print(f"- {p_instructions}: {'PASS' if res_i['passed'] else 'FAIL'}")
            if res_i["missing_headers"]:
                print("  missing headers:", ", ".join(res_i["missing_headers"]))
            if not res_i["before_after_ok"]:
                print("  missing or malformed Before/After example")
            if not res_i["tokens_present"]:
                print("  missing repo tokens (src/, tests/, pytest, mypy)")
            if res_i["forbidden_hits"]:
                print("  forbidden patterns found:", ", ".join(res_i["forbidden_hits"]))
        if not res_a["exists"]:
            print(f"- MISSING: {p_agents}")
        else:
            print(f"- {p_agents}: {'PASS' if res_a['passed'] else 'FAIL'} (agents: {res_a['agent_count']})")
            for ag in res_a["agents"]:
                tag = "PASS" if ag["passed"] else "FAIL"
                print(f"  - {ag['header']}: {tag}")
                if ag["missing_sections"]:
                    print("    missing sections:", ", ".join(ag["missing_sections"]))
                if not ag["has_template_placeholder"]:
                    print("    missing {{placeholder}} in templates")
                if not ag["has_request_and_reply"]:
                    print("    missing request/reply example")
        sys.exit(0 if report["overall_passed"] else 1)

    if __name__ == "__main__":
        main()
    """
).strip() + "\n"

p_validator = Path("tools") / "validate_agent_docs.py"
p_validator.write_text(validator_py, encoding="utf-8")
# Make it executable on POSIX (best-effort)
try:
    import os, stat
    os.chmod(p_validator, os.stat(p_validator).st_mode | stat.S_IEXEC)
except Exception:
    pass
print(f"Wrote {p_validator} ({len(validator_py.splitlines())} lines)")

What was created
- File: tools/validate_agent_docs.py — a single script that checks both artifacts and prints JSON first, then a human-readable summary. It exits non-zero if checks fail.
- How you'll reuse it: run locally during edits and add to CI so PRs fail when structure or safety checks regress.

In [ ]:
# Run the validator script and assert pass
import subprocess, json, shlex
from pathlib import Path

cmd = [sys.executable, str(Path("tools")/"validate_agent_docs.py")]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.stderr.strip():
    print("[stderr]", proc.stderr)

# Parse the first JSON object from stdout
report = None
for line in proc.stdout.splitlines():
    line = line.strip()
    if line.startswith("{") and line.endswith("}"):
        try:
            report = json.loads(line)
            break
        except json.JSONDecodeError:
            continue

assert report is not None, "Validator did not emit JSON on stdout"
assert proc.returncode == 0 and report.get("overall_passed"), "Validation failed; see summary above"
print("Overall: PASS")

Interpreting the validator's output
- The first line is machine-readable JSON; it includes per-file results and an overall_passed flag. You can consume this in other tools.
- The summary that follows points to exactly what's missing if something fails (headers, examples, tokens, or sections). Fix the noted spots and re-run.

In [ ]:
# Artifact 4 — commit the changes locally (push is optional and OFF by default)
import subprocess, os
from pathlib import Path

DO_PUSH = False  # change to True if you explicitly want to push to your remote

files_to_stage = [
    Path('.github') / 'copilot-instructions.md',
    Path('AGENTS.md'),
    Path('tools') / 'validate_agent_docs.py',
]

# Stage files
subprocess.run(["git", "add"] + [str(p) for p in files_to_stage], check=True)

# Commit if there is something to commit
status = subprocess.run(["git", "diff", "--staged", "--quiet"])  # returncode 1 means there are staged changes
if status.returncode == 1:
    subprocess.run(["git", "commit", "-m", "docs: add agent governance + validator [lesson]"], check=True)
    rev = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
    print(f"Committed as {rev}")
else:
    print("Nothing new to commit (working tree clean)")

# Optional push (requires configured remote and permissions)
if DO_PUSH:
    # Best-effort; show error if remote is missing
    try:
        subprocess.run(["git", "push"], check=True)
        print("Pushed to remote")
    except subprocess.CalledProcessError as e:
        print("Push failed (this is expected if no remote is configured):", e)

Optional CI integration (reference — illustrative, not run here)
Add a GitHub Actions workflow that runs the validator on pull requests.

```yaml
name: Validate agent docs
on:
  pull_request:
    paths:
      - '.github/copilot-instructions.md'
      - 'AGENTS.md'
      - 'tools/validate_agent_docs.py'
jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.x'
      - run: python tools/validate_agent_docs.py
```
This fails the PR if required sections/examples are missing or risky patterns are detected.

## Takeaways
- Repository-level guidance gives assistants concrete rails: where code lives (src/, tests/), how it's written (imports, typing, lint/format), and how it's proven (pytest, mypy).
- AGENTS.md clarifies who does what: named personas, allowed actions, reusable task templates with {{placeholders}}, and explicit boundaries to keep changes safe.
- A small validator script is enough to catch structure and safety regressions early; run it locally and wire it into CI so every PR stays within the guardrails.
- Workflow to remember: edit -> run tools/validate_agent_docs.py -> read report -> fix -> re-run -> git add/commit (push optional).